# 🔬 Notebook 07 — Transfer Learning ResNet50 | Experiment 01
**SATRIA DATA 2026 - Big Data Challenge**

Notebook ini merupakan orkestrasi eksperimen pertama untuk Transfer Learning menggunakan **ResNet50**.
Pendekatan: **Feature Extraction** (Backbone ImageNet di-freeze, hanya classification head yang dilatih).


In [1]:
import os
import sys
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Setup Project Root
PROJECT_ROOT = Path(os.getcwd()).resolve().parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd()).resolve()
sys.path.append(str(PROJECT_ROOT))
print(f"Project Root: {PROJECT_ROOT}")

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('ResNet50_Exp01')


c:\Users\L E N O V O\.spyder-py3\autosave\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\L E N O V O\.spyder-py3\autosave\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


ModuleNotFoundError: No module named 'torch'

---
## Stage 2 — Data Preprocessing
Menjalankan pipeline preprocessing data (Split, Augmentasi, DataLoader) sesuai konfigurasi `resnet50.yaml`.


In [ ]:
from src.preprocessing.splitter import collect_dataset, stratified_split, get_split_summary, save_split_report
from src.preprocessing.transforms import get_train_transforms, get_val_transforms
from src.preprocessing.dataloader import build_train_loader, build_val_loader, verify_dataloader

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'resnet50.yaml'
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Ambil Config
PRE_CFG = config.get('preprocessing', {})
SPL_CFG = config.get('split', {})
DL_CFG = config.get('dataloader', {})
EXP_CFG = config.get('experiment', {})

TRAIN_DIR = PROJECT_ROOT / config['data']['raw_dir'] / config['data']['train_subdir']
print(f"Data Source: {TRAIN_DIR}")


In [ ]:
# Dataset Splitting (Stratified 80/20)
df_all = collect_dataset(TRAIN_DIR)
df_train, df_val = stratified_split(
    df=df_all,
    train_ratio=SPL_CFG.get('train_ratio', 0.8),
    val_ratio=SPL_CFG.get('val_ratio', 0.2),
    random_seed=EXP_CFG.get('seed', 42)
)
print(get_split_summary(df_train, df_val))


In [ ]:
# Transforms & DataLoader
IMAGE_SIZE = PRE_CFG.get('image_size', 224)
RESIZE_TO = PRE_CFG.get('resize_to', 256)
MEAN = PRE_CFG.get('normalize', {}).get('mean', [0.485, 0.456, 0.406])
STD = PRE_CFG.get('normalize', {}).get('std', [0.229, 0.224, 0.225])
BATCH_SIZE = DL_CFG.get('batch_size', 32)
NUM_WORKERS = DL_CFG.get('num_workers', 4)

train_loader = build_train_loader(
    df_train=df_train, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    image_size=IMAGE_SIZE, resize_to=RESIZE_TO, mean=MEAN, std=STD,
    use_weighted_sampler=DL_CFG.get('use_weighted_sampler', True),
    seed=EXP_CFG.get('seed', 42)
)

val_loader = build_val_loader(
    df_val=df_val, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    image_size=IMAGE_SIZE, resize_to=RESIZE_TO, mean=MEAN, std=STD
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


---
## Stage 3 — Modeling (ResNet50 Feature Extraction)
Membangun dan melatih model. Backbone di-freeze, hanya classification head yang dilatih.


In [ ]:
from src.models.transfer_learning.resnet50 import build_resnet50, get_backbone_info
from src.training.trainer import Trainer

model = build_resnet50(config)
backbone_info = get_backbone_info(model)
for k, v in backbone_info.items():
    print(f"{k}: {v}")


In [ ]:
trainer = Trainer(model=model, config=config, project_root=PROJECT_ROOT)
print("Trainer configured successfully.")


In [ ]:
# Run Training
history = trainer.train(train_loader, val_loader)
df_history = pd.DataFrame(history)
display(df_history.tail())


In [ ]:
# Learning Curve
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(df_history['epoch'], df_history['train_loss'], label='Train')
ax[0].plot(df_history['epoch'], df_history['val_loss'], label='Val')
ax[0].set_title('Loss')
ax[0].legend()

ax[1].plot(df_history['epoch'], df_history['train_macro_f1'], label='Train F1')
ax[1].plot(df_history['epoch'], df_history['val_macro_f1'], label='Val F1')
ax[1].set_title('Macro F1')
ax[1].legend()
plt.show()


---

# Stage 4 — Evaluation: ResNet50

Pada tahap ini, kita mengevaluasi performa model **ResNet50 (Feature Extraction)** menggunakan Validation Dataset.

**Tujuan:**
1. Menghitung **Macro F1 Score** (Metrik utama BDC SATRIA DATA 2026).
2. Menganalisis *Confusion Matrix* untuk melihat error tiap kelas.
3. Memvisualisasikan *Learning Curves* untuk mendiagnosis overfitting/underfitting.
4. Melakukan *Error Analysis* pada sampel gambar.
5. Membandingkannya dengan CNN Baseline.

In [ ]:
# ============================================================
# EVALUATION STEP 1: Load Best Model Checkpoint
# ============================================================
from src.evaluation.metrics import run_inference, compute_metrics, get_classification_report, get_misclassified_samples, save_metrics
from src.evaluation.visualizer import plot_confusion_matrix, plot_learning_curves, plot_per_class_metrics, plot_error_analysis

BEST_MODEL_PATH = PROJECT_ROOT / config['output']['checkpoints_dir'] / 'resnet50_exp01_best_model.pth'

# Load Checkpoint jika sudah ada
if BEST_MODEL_PATH.exists():
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=trainer.device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Memuat best model dari Epoch {checkpoint['epoch']} dengan Val F1: {checkpoint['val_macro_f1']:.4f}")
else:
    print("⚠️ Peringatan: Checkpoint tidak ditemukan. Pastikan Anda sudah menjalankan training (Stage 3)!")

In [ ]:
# ============================================================
# EVALUATION STEP 2: Run Inference on Validation Set
# ============================================================
# Menjalankan inferensi tanpa update gradient
y_pred, y_true, y_probs = run_inference(model, val_loader, trainer.device)

print("✅ Inference selesai.")
print(f"Total Sampel: {len(y_true)}")

In [ ]:
# ============================================================
# EVALUATION STEP 3: Compute Metrics & Classification Report
# ============================================================
CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']

# Hitung Metrik Utama
metrics = compute_metrics(y_true, y_pred, class_names=CLASS_NAMES)

print("================ EVALUATION METRICS ================")
print(f"  🏆 Macro F1 Score : {metrics['macro_f1']:.4f}  <-- PRIMARY BDC METRIC")
print(f"  🎯 Accuracy       : {metrics['accuracy']:.4f}")
print(f"  🔍 Precision      : {metrics['precision']:.4f}")
print(f"  📦 Recall         : {metrics['recall']:.4f}")
print("====================================================")

# Classification Report
print("\n\ud83d\udcca CLASSIFICATION REPORT:\n")
print(get_classification_report(y_true, y_pred, class_names=CLASS_NAMES))

# Save Metrics JSON
save_metrics(metrics, PROJECT_ROOT / config['output']['reports_dir'], 'resnet50_exp01_evaluation_metrics.json')

In [ ]:
# ============================================================
# EVALUATION STEP 4: Visualizations
# ============================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

FIGURES_DIR = PROJECT_ROOT / config['output']['figures_dir']
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(
    cm=cm,
    class_names=CLASS_NAMES,
    save_path=FIGURES_DIR / 'resnet50_exp01_confusion_matrix.png',
    experiment_name='ResNet50 (Exp 01)'
)

# 2. Per-Class Metrics Bar Chart
plot_per_class_metrics(
    metrics=metrics,
    class_names=CLASS_NAMES,
    save_path=FIGURES_DIR / 'resnet50_exp01_per_class.png',
    experiment_name='ResNet50 (Exp 01)'
)

In [ ]:
# ============================================================
# EVALUATION STEP 5: Learning Curves
# ============================================================
# Pastikan history_df sudah ada di environment (dari Stage 3)
if 'df_history' in locals() or 'df_history' in globals():
    best_epoch = df_history['val_macro_f1'].idxmax() + 1
    plot_learning_curves(
        history_df=df_history,
        best_epoch=best_epoch,
        save_path=FIGURES_DIR / 'resnet50_exp01_learning_curves.png',
        experiment_name='ResNet50 (Exp 01)'
    )
else:
    print("⚠️ df_history tidak ditemukan. Run training (Stage 3) pada notebook ini terlebih dahulu.")

In [ ]:
# ============================================================
# EVALUATION STEP 6: Error Analysis (Misclassified Samples)
# ============================================================
# Ambil list of filepaths dari validation dataset (Dataset asli sebelum Dataloader)
# Karena Subset tidak punya attribute langsung 'samples', kita harus melacak indexnya.
val_indices = val_loader.dataset.indices
original_dataset = val_loader.dataset.dataset
val_filepaths = [original_dataset.samples[i][0] for i in val_indices]

misclassified_samples = get_misclassified_samples(
    y_true=y_true, y_pred=y_pred,
    filepaths=val_filepaths,
    class_names=CLASS_NAMES,
    max_samples=12
)

plot_error_analysis(
    misclassified=misclassified_samples,
    save_path=FIGURES_DIR / 'resnet50_exp01_error_analysis.png',
    experiment_name='ResNet50 (Exp 01)'
)

---

## 📝 Analisis & Kesimpulan (Template)

> *Instruksi: Jalankan semua sel di atas, lalu lengkapi bagian analisis di bawah ini berdasarkan metrik dan visualisasi yang dihasilkan.*

### 1. Interpretasi Metrik Utama
- **Accuracy vs Macro F1:** [Tulis observasi Anda...]
- **Kelas Tersulit:** Berdasarkan *Confusion Matrix*, kelas yang paling sering tertukar adalah [...] dengan [...]. Kemungkinan alasannya adalah [...]

### 2. Diagnosis Kurva Pembelajaran (Learning Curves)
- **Convergence:** Kurva Training dan Validation menunjukkan [Healthy Convergence / Overfitting / Underfitting].
- **Stabilitas:** [Tulis observasi...]

### 3. Perbandingan dengan CNN Baseline
| Metrik | CNN Baseline | ResNet50 (Exp 1) | Peningkatan |
|---|---|---|---|
| **Macro F1** | *[isi]* | *[isi]* | *[isi]* |
| **Accuracy** | *[isi]* | *[isi]* | *[isi]* |

**Analisis:** Transfer Learning ResNet50 memberikan performa [Lebih Baik / Lebih Buruk / Sama] dibandingkan model buatan manual, karena [alasan arsitektural/ekstraksi fitur].

### 4. Kesimpulan & Rekomendasi
- **Kelebihan ResNet50 pada kasus ini:** [...]
- **Kelemahan/Sisa Error:** [...]
- **Rekomendasi Selanjutnya:** [Apakah pantas lanjut ke Eksperimen 2 (Fine-Tuning)? Apakah perlu mengubah arsitektur head?]
